**The section below is for merging the basic info: CVE, outbreak, phr, enrollment for K12 students at county level**

The base data can be referenced from data folder. We will merge the data based on county name.

Notice: outbreak data other than 2025 is documented in binary way. If needs to do time series analysis, we will change the data to the actual count

In [4]:
library(tidyverse)
library(dplyr)
outbreak <- read.csv("data/count.csv")
cve <- read.csv("data/cve.csv")
phr_conv <- read.csv("data/brfss/phr_conversion.csv")
df <- cve %>%
  inner_join(outbreak, by = "County",
             suffix = c("_cve", "_out"))
head(df)

Warning message:
"Paket 'ggplot2' wurde unter R Version 4.4.3 erstellt"
Warning message:
"Paket 'forcats' wurde unter R Version 4.4.1 erstellt"
-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v dplyr     1.1.4     v readr     2.1.5
v forcats   1.0.1     v stringr   1.5.1
v ggplot2   4.0.2     v tibble    3.2.1
v lubridate 1.9.3     v tidyr     1.3.1
v purrr     1.0.2     
-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stats::lag()
i Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


,County,X2016_cve,X2017_cve,X2018_cve,X2019_cve,X2020_cve,X2021_cve,X2022_cve,X2023_cve,X2024_cve,...,X2016_out,X2017_out,X2018_out,X2019_out,X2020_out,X2021_out,X2022_out,X2023_out,X2024_out,X2025_out
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,...,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,Anderson,0.35%,0.50%,0.75%,0.94%,1.00%,1.11%,1.62%,1.64%,2.05%,...,0,0,0,0,0,0,0,0,0,0
2,Andrews,0.75%,0.90%,1.07%,1.50%,0.39%,1.36%,0.45%,1.57%,1.43%,...,0,0,0,0,0,0,0,0,0,3
3,Angelina,0.70%,0.63%,0.67%,0.82%,0.96%,0.86%,1.03%,1.53%,1.98%,...,0,0,0,0,0,0,0,0,0,0
4,Aransas,1.22%,1.39%,1.49%,1.71%,1.57%,1.51%,0.00%,1.91%,1.91%,...,0,0,0,0,0,0,0,0,0,0
5,Archer,0.61%,0.45%,0.81%,1.43%,1.29%,1.56%,1.68%,2.32%,2.80%,...,0,0,0,0,0,0,0,0,0,0
6,Armstrong,0.59%,1.90%,1.44%,2.35%,3.53%,3.53%,3.77%,4.46%,4.95%,...,0,0,0,0,0,0,0,0,0,0


In [5]:
df2025 <- df %>%
  transmute(
    County,
    cve = `X2025_cve`,
    outbreak = `X2025_out`
  )
head(df2025)

,County,cve,outbreak
,<chr>,<chr>,<int>
1,Anderson,2.54%,0
2,Andrews,1.91%,3
3,Angelina,2.50%,0
4,Aransas,2.06%,0
5,Archer,2.70%,0
6,Armstrong,5.24%,0


In [6]:
df2025$cve <- gsub("%", "", df2025$cve)
df2025$cve <- as.numeric(df2025$cve)
head(df2025)
summary(df2025)

,County,cve,outbreak
,<chr>,<dbl>,<int>
1,Anderson,2.54,0
2,Andrews,1.91,3
3,Angelina,2.50,0
4,Aransas,2.06,0
5,Archer,2.70,0
6,Armstrong,5.24,0


    County               cve            outbreak      
 Length:254         Min.   : 0.000   Min.   :  0.000  
 Class :character   1st Qu.: 1.492   1st Qu.:  0.000  
 Mode  :character   Median : 2.490   Median :  0.000  
                    Mean   : 2.817   Mean   :  2.925  
                    3rd Qu.: 3.765   3rd Qu.:  0.000  
                    Max.   :14.540   Max.   :414.000  

In [7]:
enrollment <- read.csv("data/enrollment.csv")

enrollment <- enrollment %>%
  mutate(
    County = County %>%
      str_to_lower() %>%
      str_remove(" county") %>%
      str_trim() %>%
      str_to_title()
  )
enrollment <- enrollment[enrollment$County != "Grand Total", ]
View(enrollment)

,County,Enrollment.Sum
,<chr>,<int>
1,Anderson,7808
2,Andrews,4209
3,Angelina,15649
4,Aransas,2913
5,Archer,2110
6,Armstrong,297
7,Atascosa,9046
8,Austin,6290
9,Bailey,1330


In [8]:
df2025 <- df2025 %>%
  left_join(enrollment, by = "County") %>%
  transmute(
    County,
    cve,
    outbreak,
    enrollment = Enrollment.Sum
  )


In [9]:
head(df2025)

,County,cve,outbreak,enrollment
,<chr>,<dbl>,<int>,<int>
1,Anderson,2.54,0,7808
2,Andrews,1.91,3,4209
3,Angelina,2.50,0,15649
4,Aransas,2.06,0,2913
5,Archer,2.70,0,2110
6,Armstrong,5.24,0,297


In [10]:
df2025 <- df2025 %>%
  inner_join(phr_conv, by = "County") %>%
    transmute(
      County,
      cve,
      outbreak,
      enrollment,
      phr = PHR
    )
head(df2025)
summary(df2025)

,County,cve,outbreak,enrollment,phr
,<chr>,<dbl>,<int>,<int>,<int>
1,Anderson,2.54,0,7808,4
2,Andrews,1.91,3,4209,9
3,Angelina,2.50,0,15649,5
4,Aransas,2.06,0,2913,11
5,Archer,2.70,0,2110,2
6,Armstrong,5.24,0,297,1


    County               cve            outbreak         enrollment    
 Length:254         Min.   : 0.000   Min.   :  0.000   Min.   :    93  
 Class :character   1st Qu.: 1.492   1st Qu.:  0.000   1st Qu.:  1180  
 Mode  :character   Median : 2.490   Median :  0.000   Median :  3240  
                    Mean   : 2.817   Mean   :  2.925   Mean   : 22023  
                    3rd Qu.: 3.765   3rd Qu.:  0.000   3rd Qu.: 10549  
                    Max.   :14.540   Max.   :414.000   Max.   :879002  
                                                       NA's   :5       
      phr        
 Min.   : 1.000  
 1st Qu.: 2.000  
 Median : 5.000  
 Mean   : 5.323  
 3rd Qu.: 8.000  
 Max.   :11.000  
                 

In [11]:
write.csv(df2025, file = "data/county_data.csv", row.names = FALSE)

**This section is for cleaning the general census data in texas for each county up to need**

In [ ]:
# ----- Pull all county-level variables for Texas -----
library(viridis)
library(tidycensus)
census_api_key("d57d54c0ecada0a165fe2386aa8f5078acb68d5b")
texas_counties <- get_acs(
  geography = "county",
  state     = "TX",
  year      = 2024,       
  survey    = "acs5",     
  variables = c(
    # Demographics
    pct_hispanic     = "DP05_0090PE",  # % Hispanic
    pct_black        = "DP05_0045PE",  # % Black non-Hispanic
    pct_white        = "DP05_0037PE",  # % White non-Hispanic
    
    # Socioeconomic
    pct_poverty      = "DP03_0119PE",  # % below poverty line
    pct_uninsured    = "DP03_0099PE",  # % no health insurance
    pct_college      = "DP02_0068PE",  # % college degree+
    median_income    = "DP03_0062PE",   # median household income

    
    # Rural/accessx
    pct_foreign_born = "DP02_0093PE"   # % foreign born
  ),
  output = "wide"   # puts each variable in its own column
)
# Clean it up
texas_counties <- texas_counties %>%
  # Extract county FIPS code
  mutate(county_fips = substr(GEOID, 3, 5)) %>%
  # Keep only the estimate columns (drop margin of error)
  select(GEOID, NAME, county_fips,
         pct_hispanic, pct_black, pct_white,
         pct_poverty, pct_uninsured, 
         pct_college, median_income,
         pct_foreign_born)
head(texas_counties)
summary(texas_counties)

Lade n"otiges Paket: viridisLite

Warning message:
"Paket 'tidycensus' wurde unter R Version 4.4.3 erstellt"
To install your API key for use in future sessions, run this function with `install = TRUE`.

Getting data from the 2020-2024 5-year ACS

Using the ACS Data Profile



GEOID,NAME,county_fips,pct_hispanic,pct_black,pct_white,pct_poverty,pct_uninsured,pct_college,median_income,pct_foreign_born
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
48001,"Anderson County, Texas",001,19.6,18.6,58.3,13.5,18.5,15.4,NA,1.1
48003,"Andrews County, Texas",003,57.5,1.6,57.7,14.2,22.4,17.7,NA,1.2
48005,"Angelina County, Texas",005,23.5,12.1,63.4,11.5,17.7,17.9,NA,1.2
48007,"Aransas County, Texas",007,27.1,1.3,76.9,10.7,12.8,28.8,NA,1.5
48009,"Archer County, Texas",009,9.4,1.5,88.9,4.5,13.9,25.4,NA,0.8
48011,"Armstrong County, Texas",011,12.0,0.6,89.9,6.0,4.2,27.2,NA,0.5


    GEOID               NAME           county_fips         pct_hispanic  
 Length:254         Length:254         Length:254         Min.   : 0.00  
 Class :character   Class :character   Class :character   1st Qu.:19.68  
 Mode  :character   Mode  :character   Mode  :character   Median :27.80  
                                                          Mean   :35.68  
                                                          3rd Qu.:50.02  
                                                          Max.   :97.20  
                                                                         
   pct_black        pct_white      pct_poverty    pct_uninsured  
 Min.   : 0.000   Min.   :14.40   Min.   : 0.00   Min.   : 0.00  
 1st Qu.: 1.200   1st Qu.:55.17   1st Qu.: 7.50   1st Qu.:13.90  
 Median : 3.950   Median :65.80   Median :10.80   Median :16.70  
 Mean   : 6.007   Mean   :63.84   Mean   :11.25   Mean   :16.97  
 3rd Qu.: 8.200   3rd Qu.:75.97   3rd Qu.:13.88   3rd Qu.:19.30  
 Max.   :33.

In [13]:
texas_counties <- texas_counties %>%
  mutate(
    County = NAME %>%
      str_to_lower() %>%
      str_remove(" county, texas") %>%
      str_trim() %>%
      str_to_title()
  )
head(texas_counties)

GEOID,NAME,county_fips,pct_hispanic,pct_black,pct_white,pct_poverty,pct_uninsured,pct_college,median_income,pct_foreign_born,County
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
48001,"Anderson County, Texas",001,19.6,18.6,58.3,13.5,18.5,15.4,NA,1.1,Anderson
48003,"Andrews County, Texas",003,57.5,1.6,57.7,14.2,22.4,17.7,NA,1.2,Andrews
48005,"Angelina County, Texas",005,23.5,12.1,63.4,11.5,17.7,17.9,NA,1.2,Angelina
48007,"Aransas County, Texas",007,27.1,1.3,76.9,10.7,12.8,28.8,NA,1.5,Aransas
48009,"Archer County, Texas",009,9.4,1.5,88.9,4.5,13.9,25.4,NA,0.8,Archer
48011,"Armstrong County, Texas",011,12.0,0.6,89.9,6.0,4.2,27.2,NA,0.5,Armstrong


In [14]:
texas_counties <- texas_counties[, !names(texas_counties) %in% c("GEOID", "NAME", "county_fips")]

texas_counties <- texas_counties %>% relocate(County)
head(texas_counties)
write.csv(texas_counties, file = "data/texas_census.csv", row.names = FALSE)

County,pct_hispanic,pct_black,pct_white,pct_poverty,pct_uninsured,pct_college,median_income,pct_foreign_born
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Anderson,19.6,18.6,58.3,13.5,18.5,15.4,NA,1.1
Andrews,57.5,1.6,57.7,14.2,22.4,17.7,NA,1.2
Angelina,23.5,12.1,63.4,11.5,17.7,17.9,NA,1.2
Aransas,27.1,1.3,76.9,10.7,12.8,28.8,NA,1.5
Archer,9.4,1.5,88.9,4.5,13.9,25.4,NA,0.8
Armstrong,12.0,0.6,89.9,6.0,4.2,27.2,NA,0.5


**This section is for cleaning BRFSS data.**
What we did was from the Total row, and col D10, F10,... for the response and extract as sheetname-response.

In [ ]:
"""
BRFSS Summary Tables Extractor
================================
Extracts Total + demographic breakdowns from all BRFSS PHR summary sheets
into a clean 3-sheet Excel workbook.

Usage:
    pip install openpyxl pandas
    python brfss_extract.py

Inputs / Outputs (edit the two paths below):
    SRC  – path to your source .xlsx file
    OUT  – path for the output .xlsx file
"""

import re
from openpyxl import load_workbook, Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── CONFIGURE THESE TWO PATHS ─────────────────────────────────────────────────
SRC = "data/brfss/2024_PHR1_BRFSS_Summary_Tables.xlsx"   # <- your source file
OUT = "BRFSS_1Extracted.csv"                   # <- output file
# ─────────────────────────────────────────────────────────────────────────────

# Sheets to include in the "Disease Focus" tab.
# Edit this list to match your 11 (or any number of) target sheets.
DISEASE_SHEETS = [
    "Heart Disease", "Heart Attack", "Coronary Heart Disease", "CVD", "Stroke",
    "Diabetes", "COPD", "Arthritis", "Depressive Disorder", "Kidney Disease",
    "Current Asthma", "Skin Cancer", "Other Cancer", "Any Cancer",
    "Ever Told High BP", "Fair or Poor Health", "General Health",
    "Obese", "Overweight or Obese", "BMI 3 Categories",
    "Leisure Time PA", "Current Smoker", "Heavy Drinking", "Binge Drinking",
]

# For sheets where the FIRST response column is the "healthy/good" group,
# set the dict value to the 0-based index of the column you actually want.
# Format: "Sheet Name": column_index_of_percent  (3 = first %, 5 = second %, 7 = third %)
PRIMARY_COL_OVERRIDES = {
    # Example: these sheets list "None to <5 days" first; override to the 5+ column
    "Poor Physical Health 5+ Days":  5,
    "Poor Mental Health 5+ Days":    5,
    "Hlth Affected Activ 5+ Days":   5,
    "Poor Physical Health 14+ Day":  5,
    "Poor Mental Health 14+ Days":   5,
    "Hlth Affected Activ 14+ Days":  5,
}

# Sheets to skip entirely
SKIP_SHEETS = {"Index"}


# ── Styles ────────────────────────────────────────────────────────────────────
H1         = Font(name="Arial", bold=True, size=11, color="FFFFFF")
DT         = Font(name="Arial", size=10)
BD         = Font(name="Arial", bold=True, size=10)
FILL_BLUE  = PatternFill("solid", start_color="1F4E79")  # dark blue  – header
FILL_TOTAL = PatternFill("solid", start_color="D6E4F0")  # light blue – total rows
THIN       = Side(style="thin", color="BFBFBF")
BORDER     = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

def _sc(cell, font=None, fill=None, align="left", bold=False):
    if font: cell.font  = font
    if fill: cell.fill  = fill
    cell.alignment = Alignment(horizontal=align, vertical="center", wrap_text=False)
    cell.border    = BORDER

def _hdr(ws, columns, widths):
    """Write a styled header row and set column widths."""
    for c, h in enumerate(columns, 1):
        cell = ws.cell(row=1, column=c, value=h)
        _sc(cell, font=H1, fill=FILL_BLUE, align="center")
    ws.row_dimensions[1].height = 28
    for i, w in enumerate(widths, 1):
        ws.column_dimensions[get_column_letter(i)].width = w
    ws.auto_filter.ref = f"A1:{get_column_letter(len(columns))}1"
    ws.freeze_panes    = ws.cell(row=2, column=1)


NOTE_PAT = re.compile(r"^(Note:|Software:|Prepared)", re.IGNORECASE)

def parse_sheet(ws, sheet_name):
    """
    Parse one BRFSS summary sheet and return a dict with:
        topic, age_grp, question, categories, primary_cat, primary_col, data
    Returns None if the sheet is too short to be a data sheet.
    """
    rows = list(ws.iter_rows(values_only=True))
    if len(rows) < 10:
        return None

    topic    = str(rows[2][0]).strip() if rows[2][0] else ""
    age_grp  = str(rows[3][0]).strip() if rows[3][0] else ""
    question = str(rows[4][0]).strip() if rows[4][0] else ""

    # Row 7 (0-indexed) = response category names
    # Row 8            = column sub-headers (Demographic Groups / Sample Size / % / 95% CI …)
    cat_row = rows[7]

    # Build list of (col_index, category_label) for every response category
    categories = []
    for i in range(3, len(cat_row), 2):
        v = cat_row[i]
        if v and str(v).strip() not in ("", "None"):
            categories.append((i, str(v).strip()))

    # Default: first category's % column
    default_col = categories[0][0] if categories else 3
    primary_col = PRIMARY_COL_OVERRIDES.get(sheet_name, default_col)

    # Find the matching category label for the chosen column
    primary_cat = next(
        (label for idx, label in categories if idx == primary_col),
        categories[0][1] if categories else ""
    )

    # Data rows start at index 9
    data_rows      = []
    current_group  = ""
    for r in rows[9:]:
        # Stop at blank separator or footnote rows
        if r[0] is None and r[1] is None:
            break
        if NOTE_PAT.match(str(r[0] or "")):
            break

        grp    = str(r[0]).strip() if r[0] else ""
        subgrp = str(r[1]).strip() if r[1] else ""
        if grp:
            current_group = grp

        n   = r[2]
        pct = r[primary_col]     if len(r) > primary_col     else None
        ci  = r[primary_col + 1] if len(r) > primary_col + 1 else None

        data_rows.append({
            "group":   current_group,
            "subgroup": subgrp,
            "n":        n,
            "pct":      pct,
            "ci":       ci,
        })

    return {
        "topic":       topic,
        "age_grp":     age_grp,
        "question":    question,
        "categories":  categories,
        "primary_cat": primary_cat,
        "primary_col": primary_col,
        "data":        data_rows,
    }


def build_output(parsed_all):
    wb = Workbook()

    # ── Sheet 1: Summary (one Total row per topic) ────────────────────────────
    ws1 = wb.active
    ws1.title = "Summary - Totals"
    _hdr(ws1,
         ["Sheet Name", "Topic / Category", "Age Group", "Question / Variable",
          "Primary Outcome", "Total N", "Total %", "95% CI"],
         [28, 48, 20, 60, 28, 10, 12, 20])

    for sheet_name, parsed in parsed_all.items():
        total = next((d for d in parsed["data"] if d["subgroup"] == "Total"), None)
        row   = [
            sheet_name,
            parsed["topic"],
            parsed["age_grp"],
            parsed["question"],
            parsed["primary_cat"],
            total["n"]   if total else "",
            total["pct"] if total else "",
            total["ci"]  if total else "",
        ]
        r = ws1.max_row + 1
        for c, val in enumerate(row, 1):
            cell = ws1.cell(row=r, column=c, value=val)
            _sc(cell, font=DT, fill=FILL_TOTAL)

    # ── Sheet 2: Full Detail ──────────────────────────────────────────────────
    ws2 = wb.create_sheet("Full Detail - All Demographics")
    _hdr(ws2,
         ["Sheet Name", "Topic / Category", "Primary Outcome",
          "Demographic Group", "Subgroup", "Sample N", "% (Primary)", "95% CI"],
         [28, 48, 28, 22, 30, 10, 14, 20])

    for sheet_name, parsed in parsed_all.items():
        for d in parsed["data"]:
            is_total = (d["subgroup"] == "Total")
            r        = ws2.max_row + 1
            row      = [
                sheet_name, parsed["topic"], parsed["primary_cat"],
                d["group"], d["subgroup"], d["n"], d["pct"], d["ci"],
            ]
            for c, val in enumerate(row, 1):
                cell = ws2.cell(row=r, column=c, value=val)
                _sc(cell,
                    font=BD if is_total else DT,
                    fill=FILL_TOTAL if is_total else None)

    # ── Sheet 3: Disease Focus ────────────────────────────────────────────────
    ws3 = wb.create_sheet("Disease Focus - Total + Demog")
    _hdr(ws3,
         ["Topic / Category", "Primary Outcome",
          "Demographic Group", "Subgroup", "Sample N", "% (Primary)", "95% CI"],
         [50, 28, 22, 30, 10, 14, 20])

    prev_topic = None
    for sheet_name in DISEASE_SHEETS:
        if sheet_name not in parsed_all:
            continue
        parsed = parsed_all[sheet_name]
        for d in parsed["data"]:
            is_total = (d["subgroup"] == "Total")
            is_new   = (parsed["topic"] != prev_topic)
            r        = ws3.max_row + 1
            row      = [
                parsed["topic"] if is_new else "",
                parsed["primary_cat"],
                d["group"], d["subgroup"],
                d["n"], d["pct"], d["ci"],
            ]
            for c, val in enumerate(row, 1):
                cell = ws3.cell(row=r, column=c, value=val)
                _sc(cell,
                    font=BD if is_total else DT,
                    fill=FILL_TOTAL if is_total else None)
            if is_new:
                prev_topic = parsed["topic"]
        ws3.append([""] * 7)   # blank row between topics

    return wb


def main():
    print(f"Reading: {SRC}")
    wb_in      = load_workbook(SRC, read_only=True)
    parsed_all = {}

    for sheet_name in wb_in.sheetnames:
        if sheet_name in SKIP_SHEETS:
            continue
        parsed = parse_sheet(wb_in[sheet_name], sheet_name)
        if parsed:
            parsed_all[sheet_name] = parsed

    print(f"  Parsed {len(parsed_all)} sheets")

    wb_out = build_output(parsed_all)
    wb_out.save(OUT)
    print(f"Saved:  {OUT}")
    print("Output sheets:", [s.title for s in wb_out.worksheets])


if __name__ == "__main__":
    main()

**This section is for merging all the data as a file**
The file is with base info, full BRFSS data(with total row), and census data. We will join SVI data if needed.

In [ ]:
library(tidyverse)

# Load files
county   <- read_csv("data/county_data.csv")
census   <- read_csv("data/texas_census.csv")
brfss    <- read_csv("data/brfss_dataf.csv")  # replace with actual filename

# Step 1: join census to county (both are county-level)
df <- county %>%
  left_join(census, by = "County")

# Step 2: transpose BRFSS so it's PHR-level (one row per PHR)
brfss_wide <- brfss %>%
  column_to_rownames("Sheet") %>%
  t() %>%
  as.data.frame() %>%
  mutate(across(everything(), ~ suppressWarnings(as.numeric(as.character(.))))) %>%
  rownames_to_column("phr_label") %>%
  mutate(PHR = as.integer(str_extract(phr_label, "\\d+"))) %>%
  select(-phr_label)

# Step 3: join BRFSS to county data via PHR
df_final <- df %>%
  left_join(brfss_wide, by = "PHR")

# Check
glimpse(df_final)
cat("Rows:", nrow(df_final), "| Cols:", ncol(df_final), "\n")

write_csv(df_final, "data/combined_data.csv")